# 01. Preguntas, evidencias y qrels

Las métricas de recuperación necesitan una referencia de relevancia independiente de la estrategia evaluada. Este cuaderno relaciona las formulaciones con los fragmentos juzgados y comprueba la cobertura de esa referencia. No añade ni modifica decisiones.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

In [2]:
golden = pd.DataFrame(read_jsonl(ROOT / 'data/golden/variants/generation_golden_v3.jsonl'))
qrels = pd.DataFrame(read_jsonl(ROOT / 'outputs/local_runs/explicitness_v3/qrels_final_v3.jsonl'))
chunks = pd.DataFrame(read_jsonl(ROOT / 'data/processed/chunks_v1.jsonl'))
assert len(golden) == 52 and golden['base_qid'].nunique() == 26
assert len(qrels) == qrels['unit_id'].nunique() == 578
assert not qrels.duplicated(['base_qid', 'chunk_id']).any()
assert set(qrels['base_qid']) == set(golden['base_qid'])
assert set(qrels['chunk_id']).issubset(set(chunks['chunk_id']))

## Formulación y conducta esperada

La conducta esperada distingue una respuesta ordinaria, la corrección de una premisa falsa y la abstención por falta de evidencia. Una pregunta con premisa falsa puede necesitar fragmentos que permitan refutarla; no se trata igual que una pregunta sin evidencia disponible.

In [3]:
assert golden.groupby('base_qid')['expected_behavior'].nunique().eq(1).all()
bases = golden.drop_duplicates('base_qid')
display(bases['expected_behavior'].value_counts().rename_axis('conducta').to_frame('preguntas_base'))
assert bases['expected_behavior'].value_counts().to_dict() == {'answer': 17, 'grounded_correction': 5, 'abstain': 4}
display(pd.crosstab(golden['explicitness'], golden['expected_behavior']))

,preguntas_base
conducta,
answer,17
grounded_correction,5
abstain,4


expected_behavior,abstain,answer,grounded_correction
explicitness,,,
explicit,4,17,5
implicit,4,17,5


Las dos formulaciones comparten la pregunta base, pero cambian su explicitud. La tabla permite revisar el texto exacto antes de interpretar una diferencia de rendimiento.

In [4]:
pairs = golden.pivot(index='base_qid', columns='explicitness', values='question')
assert pairs.notna().all().all()
display(pairs)

explicitness,explicit,implicit
base_qid,,
q-v1-001,How does increasing the sample size affect the sampling distribution of the sample mean?,How does increasing the sample size affect the distribution of the averages obtained b...
q-v1-002,How do Type I and Type II errors differ in terms of the truth of the null hypothesis a...,"In hypothesis testing, what are the two ways a decision about the null hypothesis can ..."
q-v1-003,What sequence of steps should be followed to conduct a hypothesis test?,What sequence of steps should be followed to evaluate a claim about a population using...
q-v1-004,Why is the median a suitable measure of center when a dataset contains extreme outliers?,Why is the middle value of the ordered dataset a suitable measure of center when a dat...
q-v1-005,What two aspects of the relationship between an independent and a dependent variable a...,What two aspects of the relationship between an independent and a dependent variable a...
q-v1-006,"If all other inputs remain unchanged, how does increasing the confidence level from 90...","If all other inputs remain unchanged, how does increasing the confidence level from 90..."
q-v1-007,"According to the selected sources, what is the Pearson correlation between weekly stud...","According to the selected sources, what value describes how strongly weekly study time..."
q-v1-008,Why does the standard error of the sample mean increase as the sample size becomes lar...,Why does the standard deviation of the sampling distribution of the mean increase as t...
q-v1-011,"For a standardized normal variable, what values define its center and spread, and how ...","For the normal distribution in its reference form, what values define its center and s..."


## Grados y procedencia de la relevancia

El grado 2 representa evidencia suficiente; el grado 1, contexto relacionado; el grado 0, material no relevante. Recall, Precision y MRR utilizan el criterio estricto de grado 2. nDCG conserva los grados 0/1/2 y calcula su ideal con toda la referencia conocida de la pregunta.

Los qrels finales se identifican por pregunta base y fragmento. El evaluador resuelve esa correspondencia para las dos formulaciones. La procedencia de cada decisión permanece en grade_source: un consenso automático no se presenta como una revisión humana.

In [5]:
display(pd.crosstab(qrels['grade_source'], qrels['grade'], margins=True))
display(qrels.groupby('pass_tag').agg(unidades=('unit_id', 'size'),
                                      preguntas_base=('base_qid', 'nunique'),
                                      fragmentos=('chunk_id', 'nunique')))

grade,0,1,2,All
grade_source,,,,
arbitration,13,15,1,29
human_validation,17,40,9,66
judge_consensus,144,253,86,483
All,174,308,96,578


,unidades,preguntas_base,fragmentos
pass_tag,,,
main,458,21,212
premise,120,5,82


La cobertura de grado 2 varía entre preguntas. Los recuentos siguientes describen la referencia disponible; no afirman que se hayan juzgado todos los fragmentos del corpus.

In [6]:
coverage = pd.crosstab(qrels['base_qid'], qrels['grade']).reindex(columns=[0, 1, 2], fill_value=0)
coverage.columns = ['grado_0', 'grado_1', 'grado_2']
coverage = coverage.join(bases.set_index('base_qid')[['expected_behavior']])
display(coverage)

,grado_0,grado_1,grado_2,expected_behavior
base_qid,,,,
q-v1-001,3,14,6,answer
q-v1-002,0,12,8,answer
q-v1-003,2,19,6,answer
q-v1-004,1,8,4,answer
q-v1-005,1,11,5,answer
q-v1-006,0,18,4,answer
q-v1-007,18,10,0,abstain
q-v1-008,3,15,7,grounded_correction
q-v1-011,9,9,2,answer


## Una evidencia completa

El ejemplo se selecciona por identificador para que pueda contrastarse con el corpus. Su grado ya está decidido en los artefactos incluidos; leerlo aquí no constituye una nueva adjudicación.

In [7]:
unit = qrels.loc[(qrels['base_qid'] == 'q-v1-001') & (qrels['grade'] == 2)].iloc[0]
text = chunks.set_index('chunk_id').loc[unit['chunk_id'], 'retrieval_text']
display(golden.loc[golden['base_qid'] == unit['base_qid'],
                   ['qid', 'question', 'reference', 'expected_behavior']])
display(unit[['base_qid', 'chunk_id', 'grade', 'grade_source', 'pass_tag']].to_frame('valor'))
print(text)

,qid,question,reference,expected_behavior
0,q-v1-001__explicit,How does increasing the sample size affect the sampling distribution of the sample mean?,"As the sample size increases, the sampling distribution of the sample mean becomes mor...",answer
1,q-v1-001__implicit,How does increasing the sample size affect the distribution of the averages obtained b...,"As the sample size increases, the sampling distribution of the sample mean becomes mor...",answer


,valor
base_qid,q-v1-001
chunk_id,openstax_intro_stats_2e:7.1:chunk-001
grade,2
grade_source,judge_consensus
pass_tag,main


Source: Introductory Statistics 2e
Chapter 7: The Central Limit Theorem
Section 7.1: The Central Limit Theorem for Sample Means (Averages)

Each time a person rolls more than one die, they calculate the sample mean of the faces showing. For example, one 
person might roll five fair dice and get 2, 2, 3, 4, 6 on one roll.

The mean is 
 = 3.4. The 3.4 is one mean when five fair dice are rolled. This same person would roll the

five dice nine more times and calculate nine more means for a total of ten means.

Your instructor will pass out the dice to several people. Roll your dice ten times. For each roll, record the faces, and 
find the mean. Round to the nearest 0.5.

Your instructor (and possibly you) will produce one graph (it might be a histogram) for one die, one graph for two 
dice, one graph for five dice, and one graph for ten dice. Since the "mean" when you roll one die is just the face on the 
die, what distribution do these means appear to be representing?

Draw the graph for

Los qrels permanecen congelados durante el reanálisis. El cuaderno siguiente utiliza esta misma referencia para todas las estrategias y comprueba los candidatos que conserva el reranking.